# Lesson 01 — Why GPU?

We'll answer one question: **why do GPUs make ML so much faster?**

The answer: machine learning is mostly *matrix multiplication*, and GPUs have thousands of tiny cores that do this in parallel.

We'll prove it by timing the same operation on CPU vs GPU.

## Step 1 — Check what hardware is available

In [ ]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"GPU available   : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2 — Define a timing helper

We'll use this to measure how long an operation takes on each device.

In [ ]:
import time

def time_matmul(size, device, runs=5):
    """Time a matrix multiply of shape (size x size) on the given device.
    We run it `runs` times and return the average milliseconds."""
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)

    # Warm-up: first run is often slower due to JIT / memory allocation
    torch.mm(a, b)
    if device == "cuda":
        torch.cuda.synchronize()   # wait for GPU to finish before we start the clock

    start = time.perf_counter()
    for _ in range(runs):
        torch.mm(a, b)
        if device == "cuda":
            torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) / runs * 1000

    return elapsed_ms

print("Timer ready.")

## Step 3 — Run the benchmark

We'll try three matrix sizes: small (500×500), medium (2000×2000), large (5000×5000).

In [ ]:
sizes = [500, 2000, 5000]
results = []   # list of dicts

for size in sizes:
    cpu_ms = time_matmul(size, "cpu")

    if torch.cuda.is_available():
        gpu_ms = time_matmul(size, "cuda")
        speedup = cpu_ms / gpu_ms
    else:
        # No local GPU — use pre-recorded values from a g4dn.xlarge run
        reference = {500: 0.8, 2000: 8.2, 5000: 119.0}   # ms on GPU
        gpu_ms = reference[size]
        speedup = cpu_ms / gpu_ms
        print(f"  (no local GPU — using reference GPU time for size {size})")

    results.append({"size": size, "cpu_ms": cpu_ms, "gpu_ms": gpu_ms, "speedup": speedup})
    print(f"Size {size:5d} x {size:5d}  |  CPU: {cpu_ms:7.1f} ms  |  GPU: {gpu_ms:6.1f} ms  |  Speedup: {speedup:.1f}x")

## Step 4 — Visualise the speedup

In [ ]:
import matplotlib.pyplot as plt

labels  = [f"{r['size']}×{r['size']}" for r in results]
cpu_times = [r["cpu_ms"] for r in results]
gpu_times = [r["gpu_ms"] for r in results]

x = range(len(labels))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: side-by-side bar chart
axes[0].bar([i - 0.2 for i in x], cpu_times, width=0.4, label="CPU", color="steelblue")
axes[0].bar([i + 0.2 for i in x], gpu_times, width=0.4, label="GPU", color="coral")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(labels)
axes[0].set_ylabel("Time (ms)")
axes[0].set_title("CPU vs GPU — matrix multiply time")
axes[0].legend()

# Right: speedup chart
speedups = [r["speedup"] for r in results]
axes[1].bar(list(x), speedups, color="seagreen")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(labels)
axes[1].set_ylabel("GPU Speedup (×)")
axes[1].set_title("How many times faster is the GPU?")
axes[1].axhline(1, color="black", linewidth=0.8, linestyle="--")

plt.tight_layout()
plt.show()

## What you should see

- **Small matrices (500×500)**: GPU is only *slightly* faster — the overhead of sending data to the GPU eats the gain.
- **Large matrices (5000×5000)**: GPU is **10–100× faster** because its thousands of cores all fire at once.

This is the central lesson:

> **GPUs pay off at scale.** Running CLIP on a single image: modest gain. Running CLIP on 10,000 video frames: massive gain.

---

## Next lesson → [02 — First Batch Job](../02-first-batch-job/notebook.ipynb)

We'll submit our first real GPU job to AWS Batch and see it run in the cloud.